In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

from skl2onnx import convert_sklearn, update_registered_converter
from onnxmltools.convert import convert_lightgbm, convert_xgboost
from skl2onnx.common.data_types import FloatTensorType, Int64TensorType
import onnxruntime as rt
import warnings
warnings.filterwarnings('ignore')

C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-ml.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-operators-ml.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\loq\AppData\Roaming\Python\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.29.3 is exactly one major version older than the runtime version 6.31.1 at onnx/onnx-data.proto. Please update the gencode to avoid compatibility violations in the next ru

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

# 1. Data Loading from CSV
print("Loading data...")
df = pd.read_csv('Dataset_Pipeline_Processed.csv')

# --- CALCULATE VOLATILITY SCORE IF MISSING ---
# The target column may not exist in the raw CSV, so we create it.
# Formula: percentage deviation of current price from its 14-day rolling average
if 'volatility_score' not in df.columns:
    print("'volatility_score' not found in CSV. Calculating from price data...")
    df['volatility_score'] = (
        (df['price_final'] - df['price_14d_avg']).abs() / df['price_14d_avg']
    ).fillna(0)
    print(f"  -> volatility_score created. Range: {df['volatility_score'].min():.4f} to {df['volatility_score'].max():.4f}")

# Convert timestamp and extract month block
df['scrape_timestamp'] = pd.to_datetime(df['scrape_timestamp'])
df['year_month'] = df['scrape_timestamp'].dt.to_period('M')

# --- DYNAMIC FEATURE SELECTION ---
target = 'volatility_score'

# Columns that should NEVER be used as features
EXCLUDED_COLUMNS = [
    'scrape_timestamp',
    'product_id',
    'raw_title',
    'year_month',
    'global_release_date_str',
    'volatility_score'
]

# Auto-detect categorical features (object dtype, not in exclusion list)
categorical_features = [
    col for col in df.columns
    if df[col].dtype == 'object' and col not in EXCLUDED_COLUMNS
]

# Auto-detect numerical features (everything else not excluded)
numerical_features = [
    col for col in df.columns
    if col not in EXCLUDED_COLUMNS and col not in categorical_features
]

features = numerical_features + categorical_features

print(f"\nDetected {len(categorical_features)} categorical features: {categorical_features}")
print(f"Detected {len(numerical_features)} numerical features: {numerical_features}")
print(f"Total features: {len(features)}")

# --- ENCODING ---
# Ordinal encode categoricals for model compatibility
if categorical_features:
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[categorical_features] = encoder.fit_transform(df[categorical_features].astype(str))

# Force all numerical features to numeric dtype (fixes 'object' columns like ram_gb_ordinal)
for col in numerical_features:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Sort strictly by product and time to prevent look-ahead
df = df.sort_values(by=['product_id', 'scrape_timestamp']).reset_index(drop=True)

print(f"\nData Shape: {df.shape}")
print(f"Time range: {df['scrape_timestamp'].min()} to {df['scrape_timestamp'].max()}")
print(f"Target '{target}' stats: mean={df[target].mean():.4f}, std={df[target].std():.4f}")

Loading data...
'volatility_score' not found in CSV. Calculating from price data...
  -> volatility_score created. Range: 0.0000 to 4.0353

Detected 5 categorical features: ['cpu_tier', 'gpu_tier', 'sale_event_label', 'ram_gb_ordinal', 'storage_ordinal']
Detected 13 numerical features: ['base_price_egp', 'official_egp_usd', 'cpi_inflation', 'is_major_sale_period', 'competitor_scarcity_count', 'import_lambda', 'volume_weight', 'missing_release_date', 'k', 'multiplier', 'price_final', 'price_14d_avg', 'D_months']
Total features: 18

Data Shape: (678410, 24)
Time range: 2025-04-16 00:00:00 to 2026-04-29 00:00:00
Target 'volatility_score' stats: mean=0.0553, std=0.1192


In [3]:
def evaluate_model_cv(df, model_class, params, fit_kwargs=None):
    """
    Evaluates a model using rolling time-series validation.
    Trains on months < T, validates on month T.
    """
    if fit_kwargs is None: fit_kwargs = {}
    
    # Drop rows where the target is missing
    clean_df = df.dropna(subset=[target])
    
    if len(clean_df) == 0:
        print(f"ERROR: No data found with target '{target}'. Check your CSV!")
        return None

    # Ensure year_month is sorted for correct temporal splitting
    months = sorted(clean_df['year_month'].unique())
    val_losses_mae = []
    
    print(f"--- Evaluating {model_class.__name__} ---")
    
    # Start loop: need at least 2 months for training to validate on the 3rd
    for val_month_idx in range(2, len(months)):
        train_months = months[:val_month_idx]
        val_month = months[val_month_idx]
        
        # Split data based on time blocks
        train_idx = clean_df[clean_df['year_month'].isin(train_months)].index
        val_idx = clean_df[clean_df['year_month'] == val_month].index
        
        if len(train_idx) == 0 or len(val_idx) == 0:
            continue
        
        X_train, y_train = clean_df.loc[train_idx, features], clean_df.loc[train_idx, target]
        X_val, y_val = clean_df.loc[val_idx, features], clean_df.loc[val_idx, target]
        
        # Initialize and train
        model = model_class(**params)
        
        # Only pass categorical_feature for LightGBM models
        current_fit_kwargs = fit_kwargs.copy()
        if 'LGBM' not in model_class.__name__ and 'categorical_feature' in current_fit_kwargs:
            del current_fit_kwargs['categorical_feature']
        
        model.fit(X_train, y_train, **current_fit_kwargs)
        
        # Predict and evaluate
        preds = model.predict(X_val)
        mae = mean_absolute_error(y_val, preds)
        val_losses_mae.append(mae)
        
        print(f"Validation Month {val_month}: MAE = {mae:.4f} ({len(val_idx)} samples)")
        
    if not val_losses_mae:
        print("No validation splits were possible. Check your date range.")
        return None
        
    avg_mae = np.mean(val_losses_mae)
    print(f"\n=> Average MAE across splits: {avg_mae:.4f}\n")
    return avg_mae

In [4]:
# --- Model 1: LightGBM ---
lgb_params = {
    'objective': 'huber',
    'alpha': 1.5,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'random_state': 42,
    'verbosity': -1,
    'n_estimators': 100
}

# Evaluate
evaluate_model_cv(
    df, 
    lgb.LGBMRegressor, 
    lgb_params, 
    fit_kwargs={'categorical_feature': categorical_features}
)

# Train Final on All Data
print("Training final LightGBM on all data...")
final_lgb = lgb.LGBMRegressor(**lgb_params)
final_lgb.fit(df[features], df[target], categorical_feature=categorical_features)
print("Done.")

--- Evaluating LGBMRegressor ---
Validation Month 2025-06: MAE = 0.0237 (53700 samples)
Validation Month 2025-07: MAE = 0.0234 (55490 samples)
Validation Month 2025-08: MAE = 0.0232 (55490 samples)
Validation Month 2025-09: MAE = 0.0246 (53700 samples)
Validation Month 2025-10: MAE = 0.0241 (55490 samples)
Validation Month 2025-11: MAE = 0.1297 (53700 samples)
Validation Month 2025-12: MAE = 0.0256 (55490 samples)
Validation Month 2026-01: MAE = 0.2840 (55490 samples)
Validation Month 2026-02: MAE = 0.0267 (50120 samples)
Validation Month 2026-03: MAE = 0.0280 (55490 samples)
Validation Month 2026-04: MAE = 0.0956 (51910 samples)

=> Average MAE across splits: 0.0644

Training final LightGBM on all data...
Done.


In [5]:
# --- Model 2: XGBoost ---
xgb_params = {
    'objective': 'reg:pseudohubererror',
    'max_depth': 6,
    'learning_rate': 0.05,
    'n_estimators': 100,
    'random_state': 42
}

# Evaluate
evaluate_model_cv(df, xgb.XGBRegressor, xgb_params)

# Train Final on All Data
print("Training final XGBoost on all data...")
final_xgb = xgb.XGBRegressor(**xgb_params)
final_xgb.fit(df[features], df[target])
print("Done.")

--- Evaluating XGBRegressor ---
Validation Month 2025-06: MAE = 0.0230 (53700 samples)
Validation Month 2025-07: MAE = 0.0226 (55490 samples)
Validation Month 2025-08: MAE = 0.0225 (55490 samples)
Validation Month 2025-09: MAE = 0.0239 (53700 samples)
Validation Month 2025-10: MAE = 0.0231 (55490 samples)
Validation Month 2025-11: MAE = 0.1319 (53700 samples)
Validation Month 2025-12: MAE = 0.0256 (55490 samples)
Validation Month 2026-01: MAE = 0.2513 (55490 samples)
Validation Month 2026-02: MAE = 0.0253 (50120 samples)
Validation Month 2026-03: MAE = 0.0265 (55490 samples)
Validation Month 2026-04: MAE = 0.0977 (51910 samples)

=> Average MAE across splits: 0.0612

Training final XGBoost on all data...
Done.


In [6]:
# --- Model 3: HistGradientBoostingRegressor (Scikit-Learn) ---
hgb_params = {
    'loss': 'absolute_error',
    'max_depth': 6,
    'learning_rate': 0.05,
    'max_iter': 100,
    'random_state': 42
}

# Evaluate
evaluate_model_cv(df, HistGradientBoostingRegressor, hgb_params)

# Train Final on All Data
print("Training final HistGradientBoosting on all data...")
final_hgb = HistGradientBoostingRegressor(**hgb_params)
final_hgb.fit(df[features], df[target])
print("Done.")

--- Evaluating HistGradientBoostingRegressor ---
Validation Month 2025-06: MAE = 0.0220 (53700 samples)
Validation Month 2025-07: MAE = 0.0216 (55490 samples)
Validation Month 2025-08: MAE = 0.0218 (55490 samples)
Validation Month 2025-09: MAE = 0.0236 (53700 samples)
Validation Month 2025-10: MAE = 0.0217 (55490 samples)
Validation Month 2025-11: MAE = 0.0931 (53700 samples)
Validation Month 2025-12: MAE = 0.0240 (55490 samples)
Validation Month 2026-01: MAE = 0.1645 (55490 samples)
Validation Month 2026-02: MAE = 0.0266 (50120 samples)
Validation Month 2026-03: MAE = 0.0283 (55490 samples)
Validation Month 2026-04: MAE = 0.0768 (51910 samples)

=> Average MAE across splits: 0.0476

Training final HistGradientBoosting on all data...
Done.


In [ ]:
# --- Final Selection & ONNX Export ---
# CHOOSE YOUR FINAL MODEL HERE based on the outputs above:
# Options: 'LightGBM', 'XGBoost', 'HistGradientBoosting'

CHOICE = 'HistGradientBoosting'  # Change this string to select

if CHOICE == 'LightGBM':
    selected_model = final_lgb
    convert_func = convert_lightgbm
elif CHOICE == 'XGBoost':
    selected_model = final_xgb
    convert_func = convert_xgboost
elif CHOICE == 'HistGradientBoosting':
    selected_model = final_hgb
    convert_func = convert_sklearn

print(f"Exporting {CHOICE} to ONNX...")

# Define Types (dynamic based on detected features)
initial_types = [('float_input', FloatTensorType([None, len(features)]))]

# Convert
onnx_model = convert_func(selected_model, initial_types=initial_types, target_opset=12)

# Save
onnx_filename = "volatility_model.onnx"
with open(onnx_filename, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model serialized to {onnx_filename}")

# Validation of ONNX Inference
sess = rt.InferenceSession(onnx_filename)
input_name = sess.get_inputs()[0].name
dummy_input = df[features].iloc[[0]].values.astype(np.float32)
pred_onnx = sess.run(None, {input_name: dummy_input})[0]

print(f"ONNX Dummy Prediction ({CHOICE}):", pred_onnx)